# TwINFER vs BEELINE benchmark - both simulation tracks

Runs the whole pipeline **from the simulation files**: TwINFER inference is executed here
(not loaded from saved results), edges are scored with Keerthana's `ka_metrics`, and the
heatmaps are drawn at the end.

**Two tracks**

| track | simulation files | notes |
|---|---|---|
| BoolODE | `simulation_data/twinfer_format/{net}/replicate_N_simulation.csv` | BoolODE model, twin-branched |
| TwINFER | `simulation_data/real_data/df_rows_*_{net}_*.csv` | TwINFER's own simulator |

**Two metrics** - `SIGNED` counts an edge correct only if direction *and* sign match;
`UNSIGNED` ignores sign. Table 17 of the report is UNSIGNED; the published heatmap is SIGNED.

**What cannot be re-run here:** GENIE3 / GRNBOOST2 / PIDC / SCODE / SCSGL are BEELINE
algorithms that run inside Docker images. There is no container runtime on this host, so
`RUN_BEELINE=False` reads their saved `rankedEdges.csv`. PEARSON and PPCOR are recomputed
from the expression matrices in this notebook regardless.

In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
import os, json, glob, re, itertools
import numpy as np, pandas as pd
from pathlib import Path
import matplotlib; import matplotlib.pyplot as plt

K   = f'{TWINFER_PROJECT_ROOT}'
# [2026-10-01 commented out: scratch was purged; the same data is in benchmarking_analysis/ (user)] NB  = '/scratch/gzu5140/twinfer_nb'          # TwINFER-sim inference outputs
NB  = f'{K}/benchmarking_analysis/twinfer_nb'          # TwINFER-sim inference outputs
# [2026-10-01 commented out: scratch was purged; the same data is in benchmarking_analysis/ (user)] TWR = '/scratch/gzu5140/twinfer_real'        # TwINFER-sim BEELINE outputs + ground truth
TWR = f'{K}/benchmarking_analysis/twinfer_real'        # TwINFER-sim BEELINE outputs + ground truth
NETS = ['GSD','HSC','VSC','mCAD']

# 2 additional real-network datasets (B_cell_activation: 10 genes, Circadian_cycle:
# 4 genes) -- simulated by TwINFER's own Gillespie simulator only (never run through
# BoolODE), so they extend the TwINFER-simulation track (6a/6c) only, not BoolODE (6b).
NEW_REAL_NETS = ['B_cell_activation', 'Circadian_cycle']
NETS_TWINFER_SIM = NETS + NEW_REAL_NETS
REAL_NET_GT_ROOT = f'{TWINFER_PROJECT_ROOT}/code/Beeline/inputs/real_networks'
REAL_NET_BEELINE_ROOT = f'{TWINFER_PROJECT_ROOT}/analysis_data/paper_analysis/real_networks/beeline_inference'

N_REPLICATES = 10    # 100 = use every replicate (matches the delivered heatmaps); lower it to run quicker
RUN_TWINFER  = True   # execute TwINFER inference here
RUN_BEELINE  = False  # needs Docker/Singularity; False = read saved rankedEdges.csv
XCORR_GATE   = 0.049
print('replicates per network:', N_REPLICATES)

## 1. Scoring functions (Keerthana's `ka_metrics`)

In [ ]:
ns = {'__name__':'ka'}
ns['INPUT_ROOT'] = Path(K+'/code/Beeline/inputs')
ns['MAX_REPS_PER_DATASET'] = 100
ns['TWINFER_JSON_ROOT'] = Path('/nonexistent')
# [2026-10-01 commented out: external file, renamed and copied into the repo as benchmark_edge_metrics.py (see docs/SCORING_VERSIONS.md)]
# exec(compile(open('/home/gzu5140/motif_sims/ka_metrics.py').read(),'ka','exec'), ns)
from twinfer.utils.paths import get_repo_root as _get_repo_root
exec(compile(open(str(_get_repo_root())+'/benchmarks/network_benchmarks/score/benchmark_edge_metrics.py').read(),'benchmark_edge_metrics','exec'), ns)
globals().update({k:v for k,v in ns.items() if not k.startswith('__')})

def f1_signed(pred, gt):
    ds = add_predicted_sign(dedupe_predictions(pred.copy()))
    ts = build_signed_edge_universe(gt)
    sel,_ = top_k_tie_aware_selection_signed(ds, len(ts))
    return precision_recall_f1(sel, ts)[2]

def f1_unsigned(pred, gt):
    ds = dedupe_predictions(pred.copy())
    ts = [(a,b) for a,b in zip(gt.Gene1, gt.Gene2)]
    ds = ds.assign(_a=ds.EdgeWeight.abs()).sort_values('_a', ascending=False)
    sel = set(map(tuple, ds[['Gene1','Gene2']].head(len(ts)).values))
    tp  = len(sel & set(ts))
    p, r = tp/max(len(sel),1), tp/max(len(ts),1)
    return 0.0 if p+r==0 else 2*p*r/(p+r)

print('scoring ready')

## 2. Load the simulation files

Both tracks are twin simulations: every cell has a partner, correlated across a time gap.

In [ ]:
def load_boolode(net, rep):
    f = f'{K}/simulation_data/twinfer_format/{net}/replicate_{rep}_simulation.csv'
    df = pd.read_csv(f)
    genes = [x.strip() for x in open(f'{K}/simulation_data/twinfer_format/{net}/gene_order.txt') if x.strip()]
    return df, genes, f

def load_twinfer_sim(net, rep):
    fs = sorted(glob.glob(f'{K}/simulation_data/real_data/*_{net}_*.csv'))
    f  = fs[rep]
    df = pd.read_csv(f)
    genes = [c[:-5] for c in df.columns if c.endswith('_mRNA')]
    return df, genes, f

df, genes, f = load_boolode('GSD', 0)
print('BoolODE  GSD rep0 :', df.shape, '|', len(genes), 'genes |', os.path.basename(f))
df2, g2, f2 = load_twinfer_sim('GSD', 0)
print('TwINFER  GSD rep0 :', df2.shape, '|', len(g2), 'genes |', os.path.basename(f2)[:60])

## 2b. Convert a TwINFER simulation into BEELINE input format

TwINFER's Gillespie simulator writes one long `df_rows_*.csv` per run (12000 cells x
5 time steps x every gene's A / I / mRNA / protein state).  BEELINE wants a wide
`ExpressionData.csv` (genes x cells), a `PseudoTime.csv`, and a
`GroundTruthNetwork.csv` edge list.  One conversion scheme is used:

| scheme | what each column is | who it is fair to |
|---|---|---|
| `twin_paired` | the same 12000 cells TwINFER itself uses (clones split 1500 / 1500 / 3000 into t1-only, t2-only, twin-1-at-t1 + twin-2-at-t2) | gives BEELINE exactly TwINFER's cells |

Gene names become `gene_1 ... gene_n` in index order of the topology matrix, so the
ground truth and the expression matrix share one name space.

*Note on the BoolODE track:* its BEELINE inputs use the same scheme
definition but were generated separately (different RNG seed, time steps 500/799),
so section 6b reads Keerthana's pre-built inputs rather than rebuilding them here.

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

# --- TwINFER simulation  ->  BEELINE input format -------------------------
# This is Keerthana's conversion (synthetic_network_analysis/twinfer_to_boolode_format.py),
# inlined here so the notebook is self-contained.  Verified in the next cell to
# reproduce /scratch/gzu5140/twinfer_real/inputs/... exactly (max |diff| = 0).
#
#   twin_paired  : the 6000 clones are split 1500 / 1500 / 3000 ->
#                    1500 clones contribute BOTH twins at t1,
#                    1500 clones contribute BOTH twins at t2,
#                    3000 clones contribute twin 1 at t1 and twin 2 at t2.
#                  That is the same split infer_with_twinfer uses internally, so
#                  the BEELINE algorithms see exactly the cells TwINFER sees.
#                  Total = 3000 + 3000 + 3000 + 3000 = 12000 columns.

T1, T2, CONV_SEED = 1, 20, 101010     # time-step indices of the TwINFER Gillespie grid

def convert_ground_truth(topology_txt_path, out_path=None):
    """Signed adjacency matrix (rows=regulators, cols=targets, -1/0/+1)
    -> BEELINE GroundTruthNetwork.csv (Gene1,Gene2,Type).  Returns (n_genes, df)."""
    m = pd.read_csv(topology_txt_path, header=None).to_numpy()
    n = m.shape[0]
    g = [f"gene_{i+1}" for i in range(n)]
    e = [(g[i], g[j], "+" if m[i, j] > 0 else "-")
         for i in range(n) for j in range(n) if m[i, j] != 0]
    gt = pd.DataFrame(e, columns=["Gene1", "Gene2", "Type"])
    if out_path is not None:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        gt.to_csv(out_path, index=False)
    return n, gt

def load_simulation(sim_csv_path, n_genes):
    """Read only what the conversion needs: ids + each gene's mRNA count."""
    mrna = [f"gene_{i+1}_mRNA" for i in range(n_genes)]
    df = pd.read_csv(sim_csv_path,
                     usecols=["cell_id", "time_step", "replicate", "clone_id"] + mrna)
    return df.rename(columns={f"gene_{i+1}_mRNA": f"gene_{i+1}" for i in range(n_genes)})

def _beeline_frames(rows, gene_names):
    """rows (long) -> (ExpressionData genes x cells, PseudoTime cells x 1)."""
    labels = [f"cell{c}_t{t}" for c, t in zip(rows["cell_id"], rows["time_step"])]
    expr = rows[gene_names].T
    expr.columns = labels
    expr.index.name = None
    pt = pd.DataFrame({"PseudoTime1": rows["time_step"].to_numpy()}, index=labels)
    return expr, pt

def build_twin_paired(df, gene_names, t1=T1, t2=T2, seed=CONV_SEED):
    rng = np.random.default_rng(seed)
    clone_ids = df["clone_id"].unique()
    assert len(clone_ids) == 6000, f"expected 6000 clones, got {len(clone_ids)}"
    sh = rng.permutation(clone_ids)
    t1_clones, t2_clones, across = sh[:1500], sh[1500:3000], sh[3000:]
    rows = pd.concat([
        df[df["clone_id"].isin(t1_clones) & (df["time_step"] == t1)],
        df[df["clone_id"].isin(t2_clones) & (df["time_step"] == t2)],
        df[df["clone_id"].isin(across) & (df["replicate"] == 1) & (df["time_step"] == t1)],
        df[df["clone_id"].isin(across) & (df["replicate"] == 2) & (df["time_step"] == t2)],
    ], ignore_index=True)
    assert len(rows) == 12000, f"expected 12000 cells, got {len(rows)}"
    return _beeline_frames(rows, gene_names)

# [2026-10-01 commented out: spread scheme removed (user); only twin_paired is used]
# def build_spread(df, gene_names, seed=CONV_SEED):
#     rng = np.random.default_rng(seed)
#     cell_ids = df["cell_id"].unique()
#     ts = np.sort(df["time_step"].unique())
#     assigned = rng.permutation(
#         np.tile(ts, int(np.ceil(len(cell_ids) / len(ts))))[:len(cell_ids)])
#     shuffled = rng.permutation(cell_ids)
#     asg = pd.DataFrame({"cell_id": shuffled, "assigned_time_step": assigned})
#     rows = df.merge(asg, on="cell_id").query("time_step == assigned_time_step")
#     assert len(rows) == len(cell_ids), f"expected {len(cell_ids)} cells, got {len(rows)}"
#     return _beeline_frames(rows, gene_names)

def write_beeline_inputs(sim_csv_path, topology_txt_path, out_dir,
# [2026-10-01 commented out: spread scheme removed (user); only twin_paired is used]                          schemes=("spread", "twin_paired")):
                         schemes=("twin_paired",)):
    """Write GroundTruthNetwork.csv once + ExpressionData/PseudoTime per scheme."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    n_genes, _ = convert_ground_truth(topology_txt_path, out_dir / "GroundTruthNetwork.csv")
    genes = [f"gene_{i+1}" for i in range(n_genes)]
    df = load_simulation(sim_csv_path, n_genes)
    made = {}
    for s in schemes:
# [2026-10-01 commented out: spread scheme removed (user); only twin_paired is used]         expr, pt = (build_spread if s == "spread" else build_twin_paired)(df, genes)
        expr, pt = build_twin_paired(df, genes)
        d = out_dir / s; d.mkdir(parents=True, exist_ok=True)
        expr.to_csv(d / "ExpressionData.csv"); pt.to_csv(d / "PseudoTime.csv")
        made[s] = expr.shape
    return made


In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
# --- check: does the conversion above reproduce the files the TwINFER-sim
#     heatmap was actually built from?  (0.0 = byte-identical numbers/labels)
# [2026-10-01 commented out: scratch was purged; the same data is in benchmarking_analysis/ (user)] _R = Path("/scratch/gzu5140/twinfer_real")
_R = Path(f"{TWINFER_PROJECT_ROOT}/benchmarking_analysis/twinfer_real")
_K = Path(f'{TWINFER_PROJECT_ROOT}')
_NET, _REP = "GSD", "117c4801"
# [2026-10-01 commented out: staged/ links point into the old Keerthana_b1042 tree and are broken; same simulation file is in simulation_data/real_data]
# _sim = next(iter((_R / "staged" / _NET).glob(f"*_{_REP}.csv")))
# [2026-10-01 fix: this glob also matches the simulation_before_division_* file (no clone_id/replicate columns) and next(iter(...)) picked it; take the df_rows_* file only]
_sim = sorted(p for p in (_K / "simulation_data" / "real_data").glob(f"df_rows_*_{_NET}_*_{_REP}.csv"))[0]
_n, _ = convert_ground_truth(_K / "input_data/real_world_networks" / f"{_NET}.txt")
_genes = [f"gene_{i+1}" for i in range(_n)]
_df = load_simulation(_sim, _n)
print(f"{_NET} rep {_REP}: {len(_df)} rows, {_n} genes")
# [2026-10-01 commented out: spread scheme removed (user); only twin_paired is used] # for _s, _b in [("twin_paired", build_twin_paired), ("spread", build_spread)]:
for _s, _b in [("twin_paired", build_twin_paired)]:
    _mine, _ = _b(_df, _genes)
    # [2026-10-01 added: the staged reference inputs of benchmarking_analysis/twinfer_real/inputs were purged with scratch (empty folders), so this self-check cannot run; skip it instead of crashing]
    _refp = _R / "inputs" / _NET / f"simrep{_REP}_{_s}" / "ExpressionData.csv"
    if not _refp.exists():
        print(f"  [{_s}] reference {_refp.name} not available (purged); conversion check skipped"); continue
    _ref = pd.read_csv(_R / "inputs" / _NET / f"simrep{_REP}_{_s}" / "ExpressionData.csv",
                       index_col=0)
    _ref.index.name = None
    _ok = list(_mine.columns) == list(_ref.columns) and list(_mine.index) == list(_ref.index)
    _d = float(np.abs(_mine.to_numpy() - _ref.to_numpy()).max()) if _ok else float("nan")
    print(f"  [{_s}] {_mine.shape} vs {_ref.shape}  same labels={_ok}  max|diff|={_d}")


## 3. Cross-correlation, computed from the simulation

`corr(gene A in twin 1 at t1, gene B in twin 2 at t2)`, Spearman, paired on clone.
This reproduces the package's `unfiltered_direction_matrix` at r = 0.9999.

In [ ]:
from scipy.stats import rankdata

def cross_correlation(df, genes, t1, t2, id_col='clone_id', twin_col='replicate',
                      time_col='time_step', suffix='_mRNA'):
    a = df[(df[twin_col]==sorted(df[twin_col].unique())[0]) & (df[time_col]==t1)].set_index(id_col).sort_index()
    b = df[(df[twin_col]==sorted(df[twin_col].unique())[1]) & (df[time_col]==t2)].set_index(id_col).sort_index()
    common = a.index.intersection(b.index); a, b = a.loc[common], b.loc[common]
    A = np.array([a[g+suffix].values for g in genes])
    B = np.array([b[g+suffix].values for g in genes])
    A = np.apply_along_axis(rankdata,1,A); B = np.apply_along_axis(rankdata,1,B)
    Ac = A - A.mean(1,keepdims=True); Bc = B - B.mean(1,keepdims=True)
    C = (Ac@Bc.T)/np.sqrt((Ac**2).sum(1)[:,None]*(Bc**2).sum(1)[None,:])
    return pd.DataFrame(C, index=genes, columns=genes), len(common)

Cm, n = cross_correlation(df, genes, 500, 799)
print(f'{n} clone pairs;  |xcorr| median = {np.median(np.abs(Cm.values[~np.eye(len(genes),dtype=bool)])):.3f}')
Cm.iloc[:4,:4].round(3)

## 4. Run TwINFER inference

The fan-out filter is patched so the two configurations in the paper can be selected:

* **BoolODE track** - `z < 137.01` flipped rule, false arms **not** eliminated
* **TwINFER track** - `z > 8`, false arms **eliminated**

Mutual inhibition is skipped in both.

In [ ]:
def fanout_decision(z, threshold, flipped):
    return (abs(z) < threshold) if flipped else (abs(z) > threshold)

def apply_fanout(ranked, fan_out_log, dmat, final_edges, threshold, flipped,
                 eliminate_false_arms, gate=XCORR_GATE):
    """Replay the fan-out stage on a ranked edge list."""
    drop = set()
    for e in (fan_out_log or []):
        A, B = e['gene_a'], e['gene_b']
        if A not in dmat.index or B not in dmat.columns: continue
        ca, cb = float(dmat.loc[A,B]), float(dmat.loc[B,A])
        if abs(ca) <= gate or abs(cb) <= gate: continue
        if ca < 0 and cb < 0: continue                      # mutual inhibition -> skip
        if eliminate_false_arms:
            Z = [z for z in (e.get('common_regulators') or [])
                 if z in dmat.index
                 and abs(float(dmat.loc[z,A])) > gate and abs(float(dmat.loc[z,B])) > gate
                 and (A,z) not in final_edges and (B,z) not in final_edges]
            if not Z: continue
        if fanout_decision(float(e['z_score']), threshold, flipped):
            drop.add(frozenset((A,B)))
    keep = ranked[[frozenset((r.Gene1,r.Gene2)) not in drop for r in ranked.itertuples()]]
    return keep, len(drop)

print('fan-out replay ready')

## 5. Benchmark methods

PEARSON and PPCOR are recomputed here from the expression matrix. The remaining five
BEELINE algorithms are read from disk unless `RUN_BEELINE=True`.

In [ ]:
def pearson_edges(X, genes):
    C = np.corrcoef(X)
    return pd.DataFrame([(genes[i],genes[j],C[i,j]) for i in range(len(genes)) for j in range(len(genes)) if i!=j],
                        columns=['Gene1','Gene2','EdgeWeight'])

def ppcor_edges(X, genes):
    C = np.corrcoef(X); P = np.linalg.pinv(C); d = np.sqrt(np.diag(P))
    Pc = -P/np.outer(d,d)
    return pd.DataFrame([(genes[i],genes[j],Pc[i,j]) for i in range(len(genes)) for j in range(len(genes)) if i!=j],
                        columns=['Gene1','Gene2','EdgeWeight'])

# [2026-10-01 commented out: spread scheme removed (user); only twin_paired is used] # def read_beeline(track, net, algo, scheme='spread'):
def read_beeline(track, net, algo, scheme='twin_paired'):
    """Yields (predictions_df, algo_dir) pairs -- algo_dir is kept (not just the parsed
    DataFrame) so the native-significance dispatch can find working_dir/outFile.txt,
    pvalues.txt, etc. relative to it."""
    if net in NEW_REAL_NETS:
        root = f'{REAL_NET_BEELINE_ROOT}/{net}'
    else:
        root = (f'{TWR}/outputs/{net}' if track=='twinfer'
                else f'{K}/analysis_data/real_networks/beeline_inference/{net}')
    out = []
    for d in sorted(glob.glob(f'{root}/simrep*_{scheme}/{algo}/rankedEdges.csv')):
        out.append((pd.read_csv(d, sep=None, engine='python'), Path(d).parent))
    return out

print('benchmark methods ready')

## 5b. Additional metric families (AUPRC, natural threshold, undirected)

Ported from `network_sweep.ipynb`. `ka_metrics.py` already provides the top-k
tie-aware and AUPRC primitives (unused here until now); the pieces genuinely new to
this notebook are the GMM natural-threshold fallback and the native-significance
dispatch (a real p-value/permutation test for PPCOR/PEARSON/SCSGL, GMM density-crossover
for everything else), plus TwINFER-side AUPRC helpers (TwINFER's ranked predictions come
from `unfiltered_direction_matrix`, not a `rankedEdges.csv`, so the BEELINE AUPRC helper
doesn't directly apply). TwINFER's own "natural threshold" is just its existing
`apply_fanout()` `keep` set scored directly -- `keep` already *is* TwINFER's hard
decision, no GMM needed.

In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
import sys
from itertools import combinations, permutations
from sklearn.mixture import GaussianMixture
from sklearn.metrics import precision_recall_curve, auc
from scipy.optimize import brentq
from scipy.stats import norm

# --- GMM natural-threshold fallback (methods with no native significance concept) ---
def fit_gmm_threshold(scores, min_points=2, random_state=0):
    """2-component GMM density-crossover threshold. Returns (gmm, threshold, mode)."""
    x = np.asarray(scores).reshape(-1, 1)
    if len(x) < min_points or len(np.unique(x)) < 2:
        return None, np.inf, "GMM_too_few_points"
    gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=random_state)
    gmm.fit(x)
    weights, means = gmm.weights_, gmm.means_.flatten()
    stds = np.sqrt(gmm.covariances_.flatten())
    order = np.argsort(means)
    w1, w2 = weights[order]; m1, m2 = means[order]; s1, s2 = stds[order]
    f = lambda z: w1 * norm.pdf(z, m1, s1) - w2 * norm.pdf(z, m2, s2)
    try:
        return gmm, brentq(f, m1, m2), "GMM_intersection"
    except ValueError:
        return gmm, np.inf, "GMM_no_intersection"

def gmm_natural_threshold_selection(predicted, threshold):
    if predicted.empty: return set()
    sel = predicted[predicted["_abs"] >= threshold]
    return set(zip(sel["Gene1"], sel["Gene2"]))

def gmm_natural_threshold_selection_signed(predicted_signed, threshold):
    if predicted_signed.empty: return set()
    sel = predicted_signed[predicted_signed["_abs"] >= threshold]
    return set(zip(sel["Gene1"], sel["Gene2"], sel["_sign"]))

def gmm_natural_threshold_selection_undirected(predicted_undirected, threshold):
    if predicted_undirected.empty: return set()
    sel = predicted_undirected[predicted_undirected["_abs"] >= threshold]
    return set(sel["_pair"])

# --- Native significance dispatch: real per-pair p-value (PPCOR/PEARSON) or
#     permutation-null threshold (SCSGL); None (-> GMM fallback) for everything else.
_SCSGL_PYSRC_DIR = Path(f'{TWINFER_PROJECT_ROOT}/code/Beeline/Algorithms/SCSGL/scSGL')
if str(_SCSGL_PYSRC_DIR) not in sys.path:
    sys.path.append(str(_SCSGL_PYSRC_DIR))
from pysrc.associations.correlation import permutations as scsgl_correlation_permutations

def expr_data_path(track, net, run_dir_name):
    """Resolves a run's ExpressionData.csv (needed by SCSGL's native permutation
    threshold), mirroring read_beeline's own per-network root branching."""
    if net in NEW_REAL_NETS:
        root = f'{REAL_NET_GT_ROOT}/{net}'
    elif track == 'twinfer':
        root = f'{TWR}/inputs/{net}'
    else:
        root = f'{K}/code/Beeline/inputs/{net}'
    return Path(root) / run_dir_name / 'ExpressionData.csv'

def _signed_undirected_from_pairs(selected_nat, predicted_signed):
    sign_lookup = dict(zip(zip(predicted_signed["Gene1"], predicted_signed["Gene2"]), predicted_signed["_sign"]))
    selected_nat_signed = {(g1, g2, sign_lookup[(g1, g2)]) for (g1, g2) in selected_nat if (g1, g2) in sign_lookup}
    selected_nat_undirected = {frozenset((g1, g2)) for (g1, g2) in selected_nat}
    return selected_nat_signed, selected_nat_undirected

def _pvalue_file_threshold_selection(algorithm, algo_dir, predicted_signed):
    """PPCOR/PEARSON: threshold on a genuine per-pair p-value file already on disk
    (PPCOR's working_dir/outFile.txt, PEARSON's working_dir/pvalues.txt)."""
    if algorithm == "PPCOR":
        pfile, alpha, pcol = algo_dir / "working_dir" / "outFile.txt", 0.01, "pValue"
    else:
        pfile, alpha, pcol = algo_dir / "working_dir" / "pvalues.txt", 0.01, "PValue"
    if not pfile.exists():
        return None
    pdf = pd.read_csv(pfile, sep="\t", header=0)
    pdf = pdf[pdf["Gene1"] != pdf["Gene2"]]
    selected_nat = set(zip(pdf.loc[pdf[pcol] < alpha, "Gene1"], pdf.loc[pdf[pcol] < alpha, "Gene2"]))
    selected_nat_signed, selected_nat_undirected = _signed_undirected_from_pairs(selected_nat, predicted_signed)
    return selected_nat, selected_nat_signed, selected_nat_undirected, alpha, f"native_pvalue_{algorithm.lower()}"

def _scsgl_permutation_selection(track, net, run_dir_name, predicted_signed, alpha=0.05, k=100):
    """SCSGL: recomputes scSGL's own (never-invoked-by-default) permutation-null
    threshold on the raw correlation matrix from that run's ExpressionData.csv."""
    expr_path = expr_data_path(track, net, run_dir_name)
    if not expr_path.exists():
        return None
    expr_df = pd.read_csv(expr_path, header=0, index_col=0)
    genes = expr_df.index.tolist()
    counts = expr_df.to_numpy()
    thresholds = scsgl_correlation_permutations(counts, k=k, tau_neg=[100 * alpha / 2], tau_pos=[100 * (1 - alpha / 2)])
    lower, upper = thresholds[0]
    corr = np.corrcoef(counts)
    n = len(genes)
    selected_nat = {(genes[i], genes[j]) for i in range(n) for j in range(n)
                     if i != j and (corr[i, j] <= lower or corr[i, j] >= upper)}
    selected_nat_signed, selected_nat_undirected = _signed_undirected_from_pairs(selected_nat, predicted_signed)
    return selected_nat, selected_nat_signed, selected_nat_undirected, (float(lower), float(upper)), "native_permutation_scsgl"

def native_threshold_for_algorithm(algorithm, algo_dir, track, net, run_dir_name, predicted_signed):
    if algorithm in ("PPCOR", "PEARSON"):
        return _pvalue_file_threshold_selection(algorithm, algo_dir, predicted_signed)
    if algorithm == "SCSGL":
        return _scsgl_permutation_selection(track, net, run_dir_name, predicted_signed)
    return None

# --- TwINFER-side AUPRC (its ranked predictions come from unfiltered_direction_matrix) ---
def build_master_ranked_table(unfiltered_direction_matrix, gene_names):
    rows = [{"gene_1": gi, "gene_2": gj, "directional_correlation": unfiltered_direction_matrix.loc[gi, gj]}
             for gi in gene_names for gj in gene_names if gi != gj]
    return pd.DataFrame(rows)

def gt_df_to_matrix(gt_df, gene_names):
    """Gene1/Gene2/Type -> signed square matrix -- the TwINFER-side AUPRC helpers need
    matrix.loc[gi,gj] access, unlike the BEELINE-side ka_metrics.py helpers which take
    gt_df directly."""
    m = pd.DataFrame(0, index=gene_names, columns=gene_names)
    for r in gt_df.itertuples():
        m.loc[r.Gene1, r.Gene2] = 1 if r.Type == '+' else -1
    return m

def compute_twinfer_auprc(ranked_edge_list, ground_truth_matrix, gene_names):
    possible_edges = list(permutations(gene_names, 2))
    true_labels = [1 if ground_truth_matrix.loc[gi, gj] != 0 else 0 for gi, gj in possible_edges]
    if sum(true_labels) == 0 or ranked_edge_list is None or ranked_edge_list.empty:
        return float("nan")
    score_lookup = {(r.gene_1, r.gene_2): abs(r.directional_correlation) for r in ranked_edge_list.itertuples()}
    pred_scores = [score_lookup.get(e, 0.0) for e in possible_edges]
    precision, recall, _ = precision_recall_curve(true_labels, pred_scores)
    return float(auc(recall, precision))

def compute_twinfer_auprc_signed(ranked_edge_list, ground_truth_matrix, gene_names):
    possible_edges = list(permutations(gene_names, 2))
    possible_signed_positions = [(gi, gj, s) for gi, gj in possible_edges for s in (1, -1)]
    true_labels = [1 if int(np.sign(ground_truth_matrix.loc[gi, gj])) == s else 0 for gi, gj, s in possible_signed_positions]
    if sum(true_labels) == 0 or ranked_edge_list is None or ranked_edge_list.empty:
        return float("nan")
    score_lookup = {(r.gene_1, r.gene_2): (abs(r.directional_correlation), int(np.sign(r.directional_correlation)))
                    for r in ranked_edge_list.itertuples()}
    pred_scores = []
    for gi, gj, s in possible_signed_positions:
        entry = score_lookup.get((gi, gj))
        pred_scores.append(entry[0] if entry is not None and entry[1] == s else 0.0)
    precision, recall, _ = precision_recall_curve(true_labels, pred_scores)
    return float(auc(recall, precision))

def compute_twinfer_auprc_undirected(ranked_edge_list, ground_truth_matrix, gene_names):
    unordered_pairs = list(combinations(gene_names, 2))
    true_labels = [1 if (ground_truth_matrix.loc[gi, gj] != 0 or ground_truth_matrix.loc[gj, gi] != 0) else 0
                   for gi, gj in unordered_pairs]
    if sum(true_labels) == 0 or ranked_edge_list is None or ranked_edge_list.empty:
        return float("nan")
    score_lookup = {}
    for r in ranked_edge_list.itertuples():
        score = abs(r.directional_correlation)
        pair = frozenset((r.gene_1, r.gene_2))
        if pair not in score_lookup or score > score_lookup[pair]:
            score_lookup[pair] = score
    pred_scores = [score_lookup.get(frozenset(p), 0.0) for p in unordered_pairs]
    precision, recall, _ = precision_recall_curve(true_labels, pred_scores)
    return float(auc(recall, precision))

# --- Per-prediction full metric suite (topk signed/unsigned/undirected, AUPRC x3, natural x3) ---
def score_beeline_prediction(d, algo_dir, algo, track, net, gt):
    # Some algorithms (small networks, degenerate variance) emit NaN EdgeWeight for a
    # handful of pairs. The top-k selectors silently tolerate this (NaN > 0 is False,
    # so those rows just never get selected), but precision_recall_curve (AUPRC) does
    # not -- sanitize once here so every downstream metric sees the same, safe input.
    d = d.copy()
    d['EdgeWeight'] = d['EdgeWeight'].fillna(0.0)

    row = {'sgn': f1_signed(d, gt), 'uns': f1_unsigned(d, gt)}
    row['auprc'] = compute_auprc(d, gt)
    row['auprc_signed'] = compute_auprc_signed(d, gt)
    row['auprc_undirected'] = compute_auprc_undirected(d, gt)

    predicted = dedupe_predictions(d)
    predicted_signed = add_predicted_sign(predicted)
    predicted_undirected = dedupe_predictions_undirected(d)

    _, true_edges = build_edge_universe(gt)
    true_edges_signed = build_signed_edge_universe(gt)
    _, true_pairs_undirected = build_undirected_edge_universe(gt)

    sel_u, _ = top_k_tie_aware_selection_undirected(predicted_undirected, len(true_pairs_undirected))
    _, _, row['uns_undirected'] = precision_recall_f1(sel_u, true_pairs_undirected)

    run_dir_name = algo_dir.parent.name
    native = native_threshold_for_algorithm(algo, algo_dir, track, net, run_dir_name, predicted_signed)
    if native is not None:
        sel_nat, sel_nat_s, sel_nat_u, _, _ = native
    else:
        _, gmm_thr, _ = fit_gmm_threshold(predicted['_abs'].values)
        sel_nat = gmm_natural_threshold_selection(predicted, gmm_thr)
        sel_nat_s = gmm_natural_threshold_selection_signed(predicted_signed, gmm_thr)
        sel_nat_u = gmm_natural_threshold_selection_undirected(predicted_undirected, gmm_thr)
    _, _, row['nat'] = precision_recall_f1(sel_nat, true_edges)
    _, _, row['nat_signed'] = precision_recall_f1(sel_nat_s, true_edges_signed)
    _, _, row['nat_undirected'] = precision_recall_f1(sel_nat_u, true_pairs_undirected)
    return row

def score_twinfer_prediction(keep, dmat, gt, gene_names):
    # Same NaN sanitization as score_beeline_prediction, for the same reason.
    keep = keep.copy()
    keep['EdgeWeight'] = keep['EdgeWeight'].fillna(0.0)
    dmat = dmat.fillna(0.0)

    row = {'sgn': f1_signed(keep, gt), 'uns': f1_unsigned(keep, gt)}

    gt_matrix = gt_df_to_matrix(gt, gene_names)
    master_table = build_master_ranked_table(dmat, gene_names)
    row['auprc'] = compute_twinfer_auprc(master_table, gt_matrix, gene_names)
    row['auprc_signed'] = compute_twinfer_auprc_signed(master_table, gt_matrix, gene_names)
    row['auprc_undirected'] = compute_twinfer_auprc_undirected(master_table, gt_matrix, gene_names)

    _, true_edges = build_edge_universe(gt)
    true_edges_signed = build_signed_edge_universe(gt)
    _, true_pairs_undirected = build_undirected_edge_universe(gt)

    keep_pairs = set(zip(keep['Gene1'], keep['Gene2']))
    keep_signed = add_predicted_sign(dedupe_predictions(keep))
    keep_signed_pairs = set(zip(keep_signed['Gene1'], keep_signed['Gene2'], keep_signed['_sign']))
    keep_undirected_pairs = {frozenset((g1, g2)) for g1, g2 in keep_pairs}

    _, _, row['nat'] = precision_recall_f1(keep_pairs, true_edges)
    _, _, row['nat_signed'] = precision_recall_f1(keep_signed_pairs, true_edges_signed)
    _, _, row['nat_undirected'] = precision_recall_f1(keep_undirected_pairs, true_pairs_undirected)

    predicted_undirected = dedupe_predictions_undirected(keep)
    sel_u, _ = top_k_tie_aware_selection_undirected(predicted_undirected, len(true_pairs_undirected))
    _, _, row['uns_undirected'] = precision_recall_f1(sel_u, true_pairs_undirected)
    return row

print('additional metric families ready')

## 6. Score everything and draw the heatmaps

In [ ]:
def heatmap(ax, M, title, vmax=0.80, nets=None):
    nets = nets if nets is not None else NETS
    n = len(nets)
    data = M[nets+['Average']].values
    im = ax.imshow(data, cmap='viridis_r', vmin=0, vmax=vmax)
    ax.set_xticks(range(n+1)); ax.set_xticklabels(nets+['Average'], rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(M))); ax.set_yticklabels(M.index, fontsize=9)
    for i in range(len(M)):
        for j in range(n+1):
            v = data[i,j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if v > 0.45 else '#111')
        ax.text(n+0.8, i, f'#{i+1}', ha='center', va='center', fontsize=9)
    ax.set_xlim(-0.5, n+1.8); ax.set_title(title, fontsize=10, pad=8)
    return im

def draw(R, banner, fname=None, nets=None):
    nets = nets if nets is not None else NETS
    fig, AX = plt.subplots(1, 2, figsize=(14.5, 6.2))
    for ax, col, lab in [(AX[0],'sgn','SIGNED  f1_topk_signed\ncorrect = direction AND sign match'),
                         (AX[1],'uns','UNSIGNED  f1_topk\ncorrect = direction only, sign ignored')]:
        M = R.groupby(['algo','net'])[col].mean().unstack()[nets]
        M['Average'] = M.mean(axis=1); M = M.sort_values('Average', ascending=False)
        im = heatmap(ax, M, lab, nets=nets)
    fig.colorbar(im, ax=AX, shrink=0.8, label='F1 (top-k, tie-aware)')
    fig.suptitle(banner, fontsize=11, fontweight='bold')
    if fname: fig.savefig(fname, dpi=140, bbox_inches='tight')
    return fig

print('plotting ready')

### 6a. TwINFER-simulation track - z > 3.2, false arms ELIMINATED

Now covers GSD/HSC/VSC/mCAD plus 2 additional real-world networks
(B_cell_activation, Circadian_cycle) -- all simulated by TwINFER's own Gillespie
simulator (never run through BoolODE, so they don't extend 6b).

In [ ]:
# [2026-10-01 fix: R_tw (TwINFER-simulation track) is computed only in the commented-out block of the original, so it was never defined and every cell that used it raised NameError. R_tw is None now and those panels are skipped; ALGOS (needed by the BoolODE track) was only defined inside that commented block]
ALGOS = ['GENIE3','GRNBOOST2','PEARSON','PIDC','PPCOR','SCODE','SCSGL']
R_tw = globals().get('R_tw')
# ALGOS = ['GENIE3','GRNBOOST2','PEARSON','PIDC','PPCOR','SCODE','SCSGL']

# # unfiltered_direction_matrix / ranked_edge_list are DataFrames; the regenerated
# # real_networks JSONs (this notebook's own make_json_safe convention -- same as
# # 6b's own JSON source, see its RAW() below) wrap them as
# # {"__type__":"DataFrame","data":...,"index":...,"columns":...} rather than a
# # plain dict-of-columns. Same unwrap idiom as 6b, defined locally here since 6b's
# # cell runs after this one.
# RAW = lambda x: pd.DataFrame(x['data'], index=x['index'], columns=x['columns'])
# def _unwrap(x):
#     return RAW(x) if isinstance(x, dict) and 'data' in x else pd.DataFrame(x)

# rows = []
# for net in NETS_TWINFER_SIM:
#     gt_path = (f'{REAL_NET_GT_ROOT}/{net}/GroundTruthNetwork.csv' if net in NEW_REAL_NETS
#                else f'{TWR}/inputs/{net}/GroundTruthNetwork.csv')
#     gt = pd.read_csv(gt_path)
#     gt = gt[gt.Gene1 != gt.Gene2].drop_duplicates()
#     for algo in ALGOS:
#         for d, algo_dir in read_beeline('twinfer', net, algo, 'twin_paired')[:N_REPLICATES]:
#             row = dict(net=net, algo=algo)
#             row.update(score_beeline_prediction(d, algo_dir, algo, 'twinfer', net, gt))
#             rows.append(row)
#     for fon in sorted(glob.glob(f'{NB}/fanout_on/{net}_rep_*_all_results.json'))[:N_REPLICATES]:
#         foff = f'{NB}/fanout_off/' + os.path.basename(fon)
#         if not os.path.exists(foff): continue
#         don, doff = json.load(open(fon)), json.load(open(foff))
#         rel = _unwrap(doff['ranked_edge_list'])
#         ranked = pd.DataFrame({'Gene1':rel['gene_1'],
#                                'Gene2':rel['gene_2'],
#                                'EdgeWeight':pd.Series(rel['directional_correlation']).astype(float)})
#         dmat = _unwrap(don['unfiltered_direction_matrix']).astype(float)
#         fde  = {tuple(x) for x in don['final_directed_edges'] if len(x)==2}
#         keep,_ = apply_fanout(ranked, don.get('fan_out_log'), dmat, fde,
#                               threshold=8.0, flipped=False, eliminate_false_arms=True)
#         row = dict(net=net, algo='TwINFER')
#         row.update(score_twinfer_prediction(keep, dmat, gt, list(dmat.index)))
#         rows.append(row)
# R_tw = pd.DataFrame(rows)
if R_tw is not None:
    draw(R_tw, 'TwINFER SIMULATION  |  TwINFER: z > 3.2, false arms ELIMINATED, mutual inhibition skipped  |  BEELINE: twin-split',
         nets=NETS_TWINFER_SIM)
    R_tw.groupby('algo')[['sgn','uns']].mean().round(4).sort_values('sgn', ascending=False)

### 6b. BoolODE track - z < 137.01 flipped, false arms NOT eliminated

In [ ]:
rows = []
for net in NETS:
    gt = load_ground_truth(net); gt = gt[gt.Gene1 != gt.Gene2].drop_duplicates()
    for algo in ALGOS:
        for d, algo_dir in read_beeline('boolode', net, algo, 'twin_paired')[:N_REPLICATES]:
            row = dict(net=net, algo=algo)
            row.update(score_beeline_prediction(d, algo_dir, algo, 'boolode', net, gt))
            rows.append(row)
    # [2026-10-01 commented out (user: not used): TwINFER rows of the BoolODE track; needed /scratch/gzu5140/ka_mi/z8_plain/*.json (fan_out_log), purged, no copy found]
    #     for rep in range(N_REPLICATES):
    #         her = f'{K}/analysis_data/boolode_sims/twinfer_inference/{net}_rep_{rep}_all_results.json'
    #         pl  = f'/scratch/gzu5140/ka_mi/z8_plain/{net}_rep_{rep}.json'
    #         if not (os.path.exists(her) and os.path.exists(pl)): continue
    #         d = json.load(open(her))
    #         RAW = lambda x: pd.DataFrame(x['data'], index=x['index'], columns=x['columns'])
    #         rel = RAW(d['ranked_edge_list']) if 'data' in d['ranked_edge_list'] else pd.DataFrame(d['ranked_edge_list'])
    #         ranked = pd.DataFrame({'Gene1':rel['gene_1'],'Gene2':rel['gene_2'],
    #                                'EdgeWeight':rel['directional_correlation'].astype(float)})
    #         dmat = RAW(d['unfiltered_direction_matrix']).astype(float)
    #         fde  = {tuple(x) for x in d['final_directed_edges'] if len(x)==2}
    #         keep,_ = apply_fanout(ranked, json.load(open(pl)).get('fan_out_log'), dmat, fde,
    #                               threshold=137.0065, flipped=True, eliminate_false_arms=False)
    #         row = dict(net=net, algo='TwINFER')
    #         row.update(score_twinfer_prediction(keep, dmat, gt, list(dmat.index)))
    #         rows.append(row)
R_bo = pd.DataFrame(rows)
draw(R_bo, 'BoolODE SIMULATION  |  TwINFER: z < 137.01 flipped, false arms NOT eliminated, mutual inhibition skipped  |  BEELINE: twin_paired')
R_bo.groupby('algo')[['sgn','uns']].mean().round(4).sort_values('sgn', ascending=False)

### 6c. Mean signed / unsigned F1 across the networks

The single number per algorithm that the heatmap shows in its **Average** column,
printed as a ranked table for both tracks. TwINFER-simulation is averaged over 6
networks (GSD/HSC/VSC/mCAD + B_cell_activation/Circadian_cycle); BoolODE stays at
the original 4.

In [ ]:
# --- 6c. Mean F1 across the networks (the "Average" column of the heatmaps)
#
# NOTE this is the mean of the per-network means, NOT a pooled mean over all
# (network, replicate) rows.  They are not the same: tracks/networks have different
# numbers of replicates (TwINFER-sim: GSD 9, HSC/VSC/mCAD/B_cell_activation/
# Circadian_cycle 10 each), so a pooled mean would silently over-weight some
# networks relative to others. The heatmaps average the networks equally, and so
# does this table.

def mean_f1_table(R, col='sgn', nets=None):
    """algo x [per-network F1, Mean], ranked by Mean (descending)."""
    nets = nets if nets is not None else NETS
    M = R.groupby(['algo', 'net'])[col].mean().unstack()[nets]
    M['Mean'] = M.mean(axis=1)
    M = M.sort_values('Mean', ascending=False)
    M.insert(0, 'rank', range(1, len(M) + 1))
    M.index.name = None
    return M.round(4)

for _R, _lab, _nets in [_x for _x in [(R_tw, 'TwINFER SIMULATION  |  z > 8, false arms ELIMINATED, '
                        'mutual inhibition skipped  |  BEELINE: twin_paired', NETS_TWINFER_SIM),
                 (R_bo, 'BoolODE SIMULATION  |  z < 137.01 flipped, false arms NOT '
                        'eliminated, mutual inhibition skipped  |  BEELINE: twin_paired', NETS)] if _x[0] is not None]:
    for _col, _name in [('sgn', 'SIGNED   f1_topk_signed  (direction AND sign must match)'),
                        ('uns', 'UNSIGNED f1_topk         (direction only, sign ignored)')]:
        print(f'\n=== {_name}\n    {_lab}')
        display(mean_f1_table(_R, _col, nets=_nets))

# --- and the same numbers as a heatmap, signed only, one panel per track.
#     (6a / 6b draw signed+unsigned side by side; this pair is the signed-only
#      layout, identical in content to the Mean tables printed above.)
_TITLES = [
    (R_tw, 'F1 Signed (top-k, tie-aware) - TwINFER simulation\n'
           'TwINFER: z > 8, false arms ELIMINATED, mutual inhibition skipped\n'
           'BEELINE: twin_paired conversion', NETS_TWINFER_SIM),
    (R_bo, 'F1 Signed (top-k, tie-aware) - BoolODE simulation\n'
           'TwINFER: z < 137.01, false arms NOT eliminated, mutual inhibition skipped', NETS),
]
_TITLES = [_t for _t in _TITLES if _t[0] is not None]   # [2026-10-01] skip the TwINFER-simulation track when R_tw is None
fig, AX = plt.subplots(1, len(_TITLES), figsize=(13.5, 5.8), squeeze=False); AX = AX[0]
for _ax, (_R, _ttl, _nets) in zip(AX, _TITLES):
    _M = mean_f1_table(_R, 'sgn', nets=_nets).drop(columns='rank').rename(columns={'Mean': 'Average'})
    _im = heatmap(_ax, _M, _ttl, nets=_nets)
fig.colorbar(_im, ax=AX, shrink=0.8, label='f1_topk_signed')
# fig.savefig('/home/gzu5140/motif_sims/heat_6c_signed.png', dpi=140, bbox_inches='tight')
plt.show()

_TITLES = [
    (R_tw, 'F1 Unsigned (top-k, tie-aware) - TwINFER simulation\n'
           'TwINFER: z > 8, false arms ELIMINATED, mutual inhibition skipped\n'
           'BEELINE: twin_paired conversion', NETS_TWINFER_SIM),
    (R_bo, 'F1 Unsigned (top-k, tie-aware) - BoolODE simulation\n'
           'TwINFER: z < 137.01, false arms NOT eliminated, mutual inhibition skipped', NETS),
]
_TITLES = [_t for _t in _TITLES if _t[0] is not None]   # [2026-10-01] skip the TwINFER-simulation track when R_tw is None
fig, AX = plt.subplots(1, len(_TITLES), figsize=(13.5, 5.8), squeeze=False); AX = AX[0]
for _ax, (_R, _ttl, _nets) in zip(AX, _TITLES):
    _M = mean_f1_table(_R, 'uns', nets=_nets).drop(columns='rank').rename(columns={'Mean': 'Average'})
    _im = heatmap(_ax, _M, _ttl, nets=_nets)
fig.colorbar(_im, ax=AX, shrink=0.8, label='f1_topk')
# fig.savefig('/home/gzu5140/motif_sims/heat_6c_unsigned.png', dpi=140, bbox_inches='tight')
plt.show()

### 6d. Additional metric families: AUPRC, natural threshold, undirected top-k

Same ranked-by-`Mean` table layout as 6c, for the metric families added in 5b.
**AUPRC** is threshold-free (ranks every possible pair, no top-k/GMM/native cutoff
involved). **Natural threshold** is each method's own hard decision: a real
significance test for PPCOR/PEARSON (p-value file) and SCSGL (permutation-null),
GMM density-crossover for the rest, and TwINFER's already-computed `apply_fanout()`
`keep` set scored directly (no further thresholding). **Undirected top-k** is the
`sgn`/`uns` top-k logic with `Gene1`/`Gene2` collapsed into one unordered pair.

In [ ]:
# --- 6d. AUPRC / natural threshold / undirected top-k -----------------------------
_EXTRA_METRICS = [
    ('auprc', 'AUPRC (directed, unsigned) -- threshold-free'),
    ('auprc_signed', 'AUPRC (directed, signed) -- threshold-free'),
    ('auprc_undirected', 'AUPRC (undirected) -- threshold-free'),
    ('nat', 'Natural threshold F1 (directed, unsigned) -- native p-value/permutation where available, GMM otherwise'),
    ('nat_signed', 'Natural threshold F1 (directed, signed)'),
    ('nat_undirected', 'Natural threshold F1 (undirected)'),
    ('uns_undirected', 'Top-k tie-aware F1 (undirected)'),
]

for _R, _lab, _nets in [_x for _x in [(R_tw, 'TwINFER SIMULATION  |  z > 8, false arms ELIMINATED, '
                        'mutual inhibition skipped  |  BEELINE: twin_paired', NETS_TWINFER_SIM),
                 (R_bo, 'BoolODE SIMULATION  |  z < 137.01 flipped, false arms NOT '
                        'eliminated, mutual inhibition skipped  |  BEELINE: twin_paired', NETS)] if _x[0] is not None]:
    for _col, _name in _EXTRA_METRICS:
        print(f'\n=== {_name}\n    {_lab}')
        display(mean_f1_table(_R, _col, nets=_nets))

In [ ]:
# --- 6d (plots). AUPRC / natural threshold / undirected top-k, heatmaps ----------
for _col, _name in _EXTRA_METRICS:
    _TITLES = [
        (R_tw, f'{_name} - TwINFER simulation\n'
               'TwINFER: z > 8, false arms ELIMINATED, mutual inhibition skipped\n'
               'BEELINE: twin_paired conversion', NETS_TWINFER_SIM),
        (R_bo, f'{_name} - BoolODE simulation\n'
               'TwINFER: z < 137.01, false arms NOT eliminated, mutual inhibition skipped', NETS),
    ]
    _TITLES = [_t for _t in _TITLES if _t[0] is not None]   # [2026-10-01] skip the TwINFER-simulation track when R_tw is None
    fig, AX = plt.subplots(1, len(_TITLES), figsize=(13.5, 5.8), squeeze=False); AX = AX[0]
    for _ax, (_R, _ttl, _nets) in zip(AX, _TITLES):
        _M = mean_f1_table(_R, _col, nets=_nets).drop(columns='rank').rename(columns={'Mean': 'Average'})
        _im = heatmap(_ax, _M, _ttl, nets=_nets)
    fig.colorbar(_im, ax=AX, shrink=0.8, label=_col)
    fig.suptitle(_name, fontsize=11, fontweight='bold')
    # fig.savefig(f'/home/gzu5140/motif_sims/heat_6d_{_col}.png', dpi=140, bbox_inches='tight')
    plt.show()

## 7. Things to try

* raise `N_REPLICATES` to 6 (BoolODE) or 9-20 (TwINFER) for the full benchmark
* in 6a/6b flip `eliminate_false_arms` and change `threshold` / `flipped` to see the filter's effect
* compare the SIGNED and UNSIGNED panels - the four sign-free methods
  (GENIE3, GRNBOOST2, PIDC, SCODE) are scored as fully wrong on sign, which is why
  they read 0.00 on VSC in the signed panel